# TP 2 - Régression Linéaire & Régression Logistique

Ce notebook couvre deux parties :
1. **Régression Linéaire** — Prédiction de la consommation de carburant (MPG) à partir du dataset Auto MPG
2. **Régression Logistique** — Prédiction de l'admission universitaire à partir du dataset Binary (UCLA)

---
## Partie 1 — Régression Linéaire

### 1.1 Import des bibliothèques

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

%matplotlib inline

### 1.2 Chargement des données CSV

On utilise le jeu de données **Auto MPG** pour prédire la consommation de carburant des automobiles (fin des années 70 / début des années 80).  
Les attributs disponibles sont : cylindres, déplacement, puissance, poids, accélération, année du modèle et origine.

In [ ]:
# Chargement du dataset avec les bons noms de colonnes
column_names = ['MPG', 'Cylinders', 'Displacement', 'Horsepower',
                'Weight', 'Acceleration', 'Model Year', 'Origin']

raw_dataset = pd.read_csv(
    'datasets/auto-mpg.csv',
    names=column_names,
    na_values='?',
    comment='\t',
    sep=' ',
    skipinitialspace=True
)

# Afficher les 5 premières lignes
raw_dataset.head()

In [ ]:
# Copie du dataset pour ne pas modifier l'original
dataSet = raw_dataset.copy()
print('Colonnes :', dataSet.columns.tolist())
print('Dimensions :', dataSet.shape)

### 1.3 Prétraitement des données

In [ ]:
# Vérification des valeurs manquantes
print('Valeurs manquantes par colonne :')
print(dataSet.isnull().sum())

In [ ]:
# Suppression des lignes avec valeurs manquantes (6 lignes dans Horsepower)
dataSet = dataSet.dropna()

print('Valeurs manquantes après suppression :')
print(dataSet.isnull().sum())
print('Nouvelles dimensions :', dataSet.shape)

### 1.4 Exploration visuelle des données

In [ ]:
# Pairplot pour visualiser les relations entre variables
%matplotlib inline
sns.pairplot(dataSet)
plt.suptitle('Pairplot - Dataset Auto MPG', y=1.02)
plt.show()

In [ ]:
# Distribution de la variable cible MPG
plt.figure(figsize=(7, 4))
sns.histplot(dataSet['MPG'], kde=True)
plt.title('Distribution de MPG')
plt.xlabel('MPG')
plt.ylabel('Fréquence')
plt.show()

In [ ]:
# Matrice de corrélation
cor = dataSet.corr()
print('Matrice de corrélation :')
print(cor)

# Heatmap de la corrélation
plt.figure(figsize=(9, 7))
sns.heatmap(cor, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Heatmap de corrélation')
plt.tight_layout()
plt.show()

> **Remarque** : Le coefficient de corrélation varie entre -1 et 1.  
> - Proche de **1** → forte corrélation positive  
> - Proche de **-1** → forte corrélation négative  
>
> Les variables **Displacement**, **Horsepower** et **Weight** montrent la plus forte corrélation (négative) avec MPG.

### 1.5 Fractionnement des données (Train / Test)

In [ ]:
# Séparation features (X) et cible (Y)
X = dataSet.drop('MPG', axis=1)
Y = dataSet['MPG']

# Split 70% entraînement / 30% test
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.3, random_state=20
)

print('Taille entraînement :', X_train.shape)
print('Taille test         :', X_test.shape)

### 1.6 Génération du modèle de Régression Linéaire

In [ ]:
# Création et entraînement du modèle
lin_model = LinearRegression()
lin_model.fit(X_train, Y_train)

print('Intercept (b0) :', lin_model.intercept_)
print('Coefficients   :', lin_model.coef_)

### 1.7 Évaluation du modèle

In [ ]:
# Prédictions sur l'ensemble de test
predictions = lin_model.predict(X_test)

# Scatter plot : valeurs réelles vs prédites
plt.figure(figsize=(7, 5))
plt.scatter(Y_test, predictions, alpha=0.6, color='steelblue')
plt.plot([Y_test.min(), Y_test.max()],
         [Y_test.min(), Y_test.max()], 'r--', lw=2, label='Parfaite prédiction')
plt.xlabel('Valeurs réelles (MPG)')
plt.ylabel('Valeurs prédites (MPG)')
plt.title('Régression Linéaire — Réel vs Prédit')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Métriques d'évaluation
mse  = mean_squared_error(Y_test, predictions)
rmse = np.sqrt(mse)
r2   = r2_score(Y_test, predictions)

count_misclassified = (Y_test != predictions).sum()

print(f'Échantillons mal classés : {count_misclassified}')
print(f'MSE  (Erreur quadratique) : {mse:.4f}')
print(f'RMSE (Racine de MSE)      : {rmse:.4f}')
print(f'R²   (Coefficient de dét.): {r2:.2f}')

---
## Partie 2 — Régression Logistique

### 2.1 Import des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing, metrics
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

%matplotlib inline

### 2.2 Chargement du jeu de données

Données de régression logistique fournies par l'**UCLA** (Institut de recherche et d'éducation numériques).  

| Colonne | Description |
|---------|-------------|
| **admit** | 1 = admis, 0 = non admis |
| **gre** | Score au Graduate Record Examination |
| **gpa** | Grade Point Average (moyenne académique) |
| **prestige** | Niveau de prestige de l'université (1 = meilleur, 4 = moins bon) |

In [ ]:
# Chargement du dataset
dataset = pd.read_csv('datasets/binary.csv')
dataset.columns = ['admit', 'gre', 'gpa', 'prestige']

print('Dimensions :', dataset.shape)
dataset.head()

In [ ]:
# Statistiques descriptives
dataset.describe()

### 2.3 Exploration visuelle

In [ ]:
# Pairplot pour visualiser les relations
sns.pairplot(dataset, hue='admit')
plt.suptitle('Pairplot - Dataset Binary (UCLA)', y=1.02)
plt.show()

### 2.4 Prétraitement des données

In [ ]:
# Vérification des valeurs manquantes
print('Valeurs manquantes :')
print(dataset.isnull().sum())

In [ ]:
# Encodage one-hot de la variable catégorielle 'prestige'
dummy_ranks = pd.get_dummies(dataset['prestige'], prefix='prestige')
dummy_ranks.head()

In [ ]:
# Construction du DataFrame final pour la régression
cols = ['admit', 'gre', 'gpa']
data = dataset[cols].join(dummy_ranks)

print('Répartition de la variable cible (admit) :')
print(data['admit'].value_counts())
data.head()

### 2.5 Fractionnement des données (Train / Test)

In [ ]:
# Séparation features et cible
x = data.drop('admit', axis=1)
y = data['admit']

# Split 70% / 30%
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=1
)

print('Taille entraînement :', x_train.shape)
print('Taille test         :', x_test.shape)

### 2.6 Génération du modèle de Régression Logistique

In [ ]:
# Création et entraînement du modèle
logmodel = LogisticRegression(C=1e20, max_iter=1000)
logmodel.fit(x_train, y_train)

print('Intercept (b0) :', logmodel.intercept_)
print('Coefficients   :', logmodel.coef_)

La fonction score obtenue est :

$$score(X) = b_0 + b_1 \cdot gre + b_2 \cdot gpa + b_3 \cdot prestige\_1 + b_4 \cdot prestige\_2 + b_5 \cdot prestige\_3 + b_6 \cdot prestige\_4$$

### 2.7 Évaluation du modèle

In [ ]:
# Prédictions sur l'ensemble de test
y_pred = logmodel.predict(x_test)
print('Prédictions :', y_pred)

In [ ]:
# Métriques de performance
count_misclassified = (y_test != y_pred).sum()
accuracy = metrics.accuracy_score(y_test, y_pred)

print(f'Échantillons mal classés : {count_misclassified}')
print(f'Précision (Accuracy)     : {accuracy:.2f}')

In [ ]:
# Rapport de classification détaillé
print('Rapport de classification :')
print(metrics.classification_report(y_test, y_pred,
      target_names=['Non admis (0)', 'Admis (1)']))

In [ ]:
# Matrice de confusion
cm = metrics.confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non admis', 'Admis'],
            yticklabels=['Non admis', 'Admis'])
plt.xlabel('Prédiction')
plt.ylabel('Réel')
plt.title('Matrice de Confusion — Régression Logistique')
plt.tight_layout()
plt.show()

---
## Conclusion

| Modèle | Dataset | Métrique | Résultat |
|--------|---------|----------|----------|
| Régression Linéaire | Auto MPG | R² | ~0.84 |
| Régression Logistique | Binary (UCLA) | Accuracy | ~0.74 |

- La **régression linéaire** prédit bien la consommation MPG (R² ≈ 0.84).
- La **régression logistique** classe correctement ~74% des admissions universitaires.